# Train and Evaluate a Policy Using the Time2Success Model as Reward

Pure model-derived reward only — no dense reward, no ground-truth bonus.
Uses the retrained time2success model (checkpoint-diverse + uncapped
post-success collection, confirmed working via the 10-seed prediction check).

Sections: setup → load frozen model → reward wrapper → train (smoke test
first) → evaluate the resulting policy → compare against stage 1.

## 1. Setup

In [2]:
import numpy as np
import torch
import torch.nn as nn
import gymnasium as gym
import metaworld
import imageio
import collections
import json, os, time
from stable_baselines3 import SAC
from stable_baselines3.common.vec_env import DummyVecEnv, VecMonitor
from stable_baselines3.common.callbacks import BaseCallback

TASK_NAME = "peg-insert-side-v3"
SUCCESS_KEY = "success"

def make_env(seed=0, render_mode=None):
    return gym.make("Meta-World/MT1", env_name=TASK_NAME, seed=seed, render_mode=render_mode, camera_name = 'corner2')

_probe = make_env(seed=0)
DT = getattr(_probe.unwrapped, "dt", _probe.unwrapped.model.opt.timestep)
OBS_DIM = _probe.observation_space.shape[0]
_probe.close()
print(f"DT={DT}, OBS_DIM={OBS_DIM}")

class Time2SuccessModel(nn.Module):
    def __init__(self, obs_dim, hidden=256, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

device = "cuda" if torch.cuda.is_available() else "cpu"
# No dataset loading here — this notebook only consumes the ALREADY-TRAINED
# time2success model, loaded explicitly in the next cell.

DT=0.0125, OBS_DIM=39


d:\Miniconda3\envs\duke_rob\lib\site-packages\gymnasium\utils\passive_env_checker.py:34: UserWarning: WARN: A Box observation space maximum and minimum values are equal.
  logger.warn("A Box observation space maximum and minimum values are equal.")


## 2. Load the frozen, retrained time2success model

In [2]:
norm = np.load("./checkpoints/time2successV4/normalization.npz")
SHAPING_X_MEAN, SHAPING_X_STD = norm["X_mean"], norm["X_std"]
SHAPING_Y_MEAN, SHAPING_Y_STD = norm["y_mean"].item(), norm["y_std"].item()

shaping_t2s_model = Time2SuccessModel(obs_dim=OBS_DIM).to(device)
shaping_t2s_model.load_state_dict(
    torch.load("./checkpoints/time2successV4/time2success_state_model_best.pt"))
shaping_t2s_model.eval()  # frozen
print("Loaded frozen time2success model + normalization stats")
print("Confirm this is the retrained (checkpoint-diverse, uncapped-window) version, "
      "not an earlier checkpoint of the model file.")

Loaded frozen time2success model + normalization stats
Confirm this is the retrained (checkpoint-diverse, uncapped-window) version, not an earlier checkpoint of the model file.


## 3. Pure reward wrapper (only variant kept — additive-shaped and diff+bonus variants removed)

In [3]:
GAMMA = 0.99

class PureTime2SuccessRewardWrapper(gym.Wrapper):
    def __init__(self, env, t2s_model, device, x_mean, x_std, y_mean, y_std,
                 gamma=GAMMA, shaping_scale=1.0, stall_window=20,
                 max_pred_steps=500):
        super().__init__(env)
        self.t2s_model = t2s_model
        self.device = device
        self.x_mean, self.x_std = x_mean, x_std
        self.y_mean, self.y_std = y_mean, y_std
        self.gamma = gamma
        self.shaping_scale = shaping_scale
        self.stall_window = stall_window
        self.max_pred_steps = max_pred_steps  # output clipping safety net
        self._last_phi = 0.0
        self._recent_predictions = collections.deque(maxlen=self.stall_window)

    @torch.no_grad()
    def _potential(self, obs):
        x_norm = (obs - self.x_mean) / self.x_std
        x = torch.tensor(x_norm, dtype=torch.float32).unsqueeze(0).to(self.device)
        pred_norm = self.t2s_model(x).item()
        pred_steps = pred_norm * self.y_std + self.y_mean
        pred_steps = float(np.clip(pred_steps, 0, self.max_pred_steps))  # bound, per earlier discussion
        return -pred_steps

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        self._last_phi = self._potential(obs)
        self._recent_predictions.clear()
        self._recent_predictions.append(-self._last_phi)
        return obs, info

    def step(self, action):
        obs, _base_reward, terminated, truncated, info = self.env.step(action)
        phi_next = self._potential(obs)
        predicted_now = -phi_next

        if (len(self._recent_predictions) == self.stall_window
                and predicted_now >= max(self._recent_predictions)):
            truncated = True

        self._recent_predictions.append(predicted_now)
        # Reward shaping based on time-to-success potential difference
        pure_t2s_reward = self.gamma * phi_next - self._last_phi
        self._last_phi = phi_next
        total_reward = self.shaping_scale * pure_t2s_reward
        return obs, total_reward, terminated, truncated, info

def make_pure_t2s_env(seed=0):
    base_env = make_env(seed=seed)
    return PureTime2SuccessRewardWrapper(
        base_env, shaping_t2s_model, device,
        SHAPING_X_MEAN, SHAPING_X_STD, SHAPING_Y_MEAN, SHAPING_Y_STD,
    )

## 4. Build training setup + callback (consistently `pure_*`, no leftover `shaped_*` references)

In [4]:
N_ENVS = 6
pure_train_env = DummyVecEnv([lambda i=i: make_pure_t2s_env(seed=i) for i in range(N_ENVS)])
pure_train_env = VecMonitor(pure_train_env)
pure_eval_env = make_env(seed=1000)

pure_model = SAC(
    policy="MlpPolicy",
    env=pure_train_env,
    learning_rate=3e-4,
    buffer_size=1_000_000,
    batch_size=256,
    tau=0.005,
    gamma=GAMMA,
    ent_coef="auto",
    policy_kwargs=dict(net_arch=[400, 400]),
    tensorboard_log="./tb_logs/peg_insert_side_pure_t2s",
    verbose=1,
    seed=0,
)

Using cpu device


In [5]:
class SuccessCallback(BaseCallback):
    def __init__(self, eval_env, eval_freq=10_000, n_eval_episodes=10,
                 ckpt_dir="./checkpoints/peg_insert_side_pure_t2s", ckpt_freq=100_000, verbose=1):
        super().__init__(verbose)
        self.eval_env = eval_env
        self.eval_freq = eval_freq
        self.n_eval_episodes = n_eval_episodes
        self.ckpt_dir = ckpt_dir
        self.ckpt_freq = ckpt_freq
        os.makedirs(ckpt_dir, exist_ok=True)
        self.history = []

    def _run_eval_episode(self):
        obs, _ = self.eval_env.reset()
        success_step = None
        for t in range(500):
            action, _ = self.model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = self.eval_env.step(action)
            if info.get(SUCCESS_KEY, 0) and success_step is None:
                success_step = t
            if terminated or truncated:
                break
        return success_step

    def _on_step(self) -> bool:
        if self.num_timesteps % self.ckpt_freq < self.training_env.num_envs:
            self.model.save(os.path.join(self.ckpt_dir, f"pure_t2s_{self.num_timesteps}.zip"))
        if self.num_timesteps % self.eval_freq < self.training_env.num_envs:
            steps = [self._run_eval_episode() for _ in range(self.n_eval_episodes)]
            success_rate = sum(s is not None for s in steps) / self.n_eval_episodes
            times = [s for s in steps if s is not None]
            mean_t2s = float(np.mean(times)) if times else None
            self.logger.record("eval/success_rate", success_rate)
            if mean_t2s is not None:
                self.logger.record("eval/mean_time_to_success", mean_t2s)
            self.history.append(dict(step=self.num_timesteps, success_rate=success_rate,
                                       mean_time_to_success=mean_t2s, timestamp=time.time()))
            with open(os.path.join(self.ckpt_dir, "eval_history.json"), "w") as f:
                json.dump(self.history, f, indent=2)
            if self.verbose:
                print(f"[eval @ {self.num_timesteps}] success_rate={success_rate:.2f} mean_t2s={mean_t2s}")
        return True

pure_callback = SuccessCallback(eval_env=pure_eval_env, ckpt_freq=2000, eval_freq=2000)

## 5. Smoke test

In [ ]:
pure_model.learn(total_timesteps=5_000, callback=pure_callback, tb_log_name="pure_t2s_smoke")
print("Smoke test complete — check eval_history.json and eval/success_rate output above")

## 6. Full run — only after the smoke test looks reasonable

In [ ]:
TOTAL_TIMESTEPS = 3_000_000

pure_model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    callback=pure_callback,
    tb_log_name="pure_t2s_run2",
    progress_bar=True,
    reset_num_timesteps=False,
)
pure_model.save("./checkpoints/peg_insert_side_pure_t2s/sac_peg_insert_pure_t2s_final")
pure_train_env.close()
pure_eval_env.close()
print("Pure time2success-reward training complete")

Logging to ./tb_logs/peg_insert_side_pure_t2s\pure_t2s_run2_0


Output()

## 7. Evaluate the trained policy — multi-seed success check + rollout video

Two visual verifications kept, as requested: (a) a quick numeric success-rate
check across several seeds, and (b) an actual rollout video to watch.

In [3]:
eval_policy = SAC.load("./checkpoints/peg_insert_side_ensemble_t2s/ens_t2s_500004")

N_EVAL = 20
success_steps = []
for seed in range(N_EVAL):
    env = make_env(seed=seed)
    obs, _ = env.reset()
    success_step = None
    for t in range(500):
        action, _ = eval_policy.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        if info.get(SUCCESS_KEY, 0) and success_step is None:
            success_step = t
        if terminated or truncated:
            break
    env.close()
    success_steps.append(success_step)

success_rate = np.mean([s is not None for s in success_steps])
times = [s for s in success_steps if s is not None]
print(f"Success rate over {N_EVAL} seeds: {success_rate:.2%}")
if times:
    print(f"Mean time-to-success: {np.mean(times):.1f} steps")

d:\Miniconda3\envs\duke_rob\lib\site-packages\gymnasium\utils\passive_env_checker.py:34: UserWarning: WARN: A Box observation space maximum and minimum values are equal.
  logger.warn("A Box observation space maximum and minimum values are equal.")
d:\Miniconda3\envs\duke_rob\lib\site-packages\gymnasium\utils\passive_env_checker.py:157: UserWarning: WARN: The obs returned by the `reset()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")
d:\Miniconda3\envs\duke_rob\lib\site-packages\gymnasium\utils\passive_env_checker.py:157: UserWarning: WARN: The obs returned by the `step()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")


Success rate over 20 seeds: 0.00%


In [4]:
def record_rollout(model, env, out_path, max_steps=500, deterministic=True):
    obs, _ = env.reset()
    frames = []
    success_step = None
    for t in range(max_steps):
        frames.append(env.render())
        action, _ = model.predict(obs, deterministic=deterministic)
        obs, reward, terminated, truncated, info = env.step(action)
        if info.get(SUCCESS_KEY, 0) and success_step is None:
            success_step = t
        if terminated or truncated:
            break
    imageio.mimsave(out_path, frames, fps=20)
    return success_step, len(frames)

render_env = make_env(seed=0, render_mode="rgb_array")
success_step, num_frames = record_rollout(eval_policy, render_env, "pure_t2s_policy_behavior.mp4")
render_env.close()
print(f"Rollout saved. success_step={success_step}, num_frames={num_frames}")

Rollout saved. success_step=None, num_frames=500


In [5]:
from IPython.display import Video
Video("pure_t2s_policy_behavior.mp4", embed=True)

## 8. Compare against stage 1 — the real verdict

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

with open("./checkpoints/peg_insert_side/eval_history.json") as f:
    orig_history = json.load(f)
with open("./checkpoints/peg_insert_side_pure_t2s/eval_history.json") as f:
    pure_history = json.load(f)

orig_df = pd.DataFrame(orig_history)
pure_df = pd.DataFrame(pure_history)

plt.figure(figsize=(7,4))
plt.plot(orig_df["step"], orig_df["success_rate"], label="original (dense reward)")
plt.plot(pure_df["step"], pure_df["success_rate"], label="pure time2success reward")
plt.xlabel("step")
plt.ylabel("success rate")
plt.legend()
plt.title("Original vs. pure time2success-only reward")
plt.show()